In [ ]:
from datasets import load_dataset
import pandas as pd
import re
from huggingface_hub import login
import os
from transformers import MarianMTModel, MarianTokenizer, BertTokenizer, AutoTokenizer
import torch
import gc
from tqdm import tqdm

tqdm.pandas()

rseed = 42
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
############ Masking
PLACEHOLDER = "John"

def protect_mbti_mask(text):
    text = text.replace("<mbti>", PLACEHOLDER)
    return text

def restore_mbti_mask(text):
    text = re.sub(rf"{PLACEHOLDER}", "<mbti>", text, flags=re.IGNORECASE)
    return text
#################

In [ ]:
df_hfdict = load_dataset("DrinkIcedT/mbti_unbalanced")
df = df_hfdict["train"].to_pandas()

In [ ]:
long_posts = df[df["post"].str.split().str.len() >= 5]
df_subset = long_posts[long_posts["labels"] == 9]
df_i = long_posts[long_posts["I"] == 1]

In [ ]:
print(df_i)

# Class Distributions
Looking at the class distribution, it becomes obvious, that the biggest imbalance is in N/S dimension, which is roughly 7:1 while the next biggest is the I/E dimension with 3:1. For this reason and to ensure low quality degradation, oversampling should only be limited to the N/S dimension

In [ ]:
print(long_posts["I"].value_counts())
print(long_posts["N"].value_counts())
print(long_posts["F"].value_counts())
print(long_posts["P"].value_counts())

In [ ]:
df_dim_S = df[df["N"] == 0]
#print(df["N"].value_counts())

print(df["N"][df["N"] == 0].count())

In [ ]:
def dim_samples(df, label, value, n_goal, rseed):
    # count orginals
    obs_count = df["N"][df["N"] == 0].count()
    
    if obs_count < n_goal:
        n_diff = n_goal - obs_count
        
        # only text longer than 5 chars
        long_posts = df[df["post"].str.split().str.len() >= 5]
        df_subset = long_posts[long_posts[label] == value]

        # fallback
        if df_subset.empty:
            df_subset = df[df[label] == value]

        sample = df_subset.sample(n=n_diff, replace=True, random_state=rseed)

        return sample
    else:
        empty = df.iloc[:0]
        return empty

In [ ]:
#N_sample = pd.DataFrame(columns=df.columns)

N_sample = dim_samples(df, "N", 0, 10000, rseed)
print(len(N_sample))

# for label in df["labels"].unique():
#     s1, s2, s3, s4 = label_samples(df, label, n_goal=2000, rseed=42)
#     syn_sample = pd.concat([syn_sample, s1])
#     bt_sample = pd.concat([bt_sample, s2])
#     sw_sample = pd.concat([sw_sample, s3])
#     del_sample = pd.concat([del_sample, s4])



# Dropna
#N_sample = N_sample.dropna(subset=["post"])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# Anzahl der Tokens pro Dokument
N_sample["token_count"] = N_sample["post"].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True))
)

print((N_sample["token_count"] > 512).sum())

print(N_sample["token_count"].describe())

In [ ]:
###### Backtranslation funtion
def backtranslate_safe(texts, device="cuda"):
    # load models
    en_de_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
    en_de_mod = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-de").to(device)
    de_en_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-de-en")
    de_en_mod = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-de-en").to(device)

    gen_kwargs = dict(
        top_p=0.95,
        temperature=0.5,
        repetition_penalty=1.1,
        no_repeat_ngram_size=3,
        max_length=512,
        do_sample=True,
    )

    def translate(model, tokenizer, batch):
        if not batch: return []
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                          truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, **gen_kwargs)
        return tokenizer.batch_decode(out, skip_special_tokens=True)

    #### split texts
    all_sentences_to_translate = []
    group_sizes = []

    for text in texts:
        protected_text = protect_mbti_mask(text)
        parts = [p.strip() for p in protected_text.split("</s>") if p.strip()]
        all_sentences_to_translate.extend(parts)
        group_sizes.append(len(parts))

    #### batch translation
    translated_results = []
    batch_size = 16
    
    print(f"Übersetze {len(all_sentences_to_translate)} Einzel-Posts...")
    for i in tqdm(range(0, len(all_sentences_to_translate), batch_size)):
        batch = all_sentences_to_translate[i:i+batch_size]
        
        # backtranslation
        de = translate(en_de_mod, en_de_tok, batch)
        en_back = translate(de_en_mod, de_en_tok, de)
        
        translated_results.extend(en_back)

    # empty cache
    del en_de_mod, de_en_mod
    gc.collect()
    torch.cuda.empty_cache()

    #### build back together grouped by author
    final_texts = []
    current_idx = 0
    for size in group_sizes:
        group = translated_results[current_idx : current_idx + size]
        combined = " </s> ".join(group)
        # placeholder
        final_texts.append(restore_mbti_mask(combined))
        current_idx += size

    return final_texts

In [ ]:
texts = N_sample["post"].tolist()
print(len(texts))
print(texts[1])
#N_sample["post_augmented"] = backtranslate_safe(texts)

# safe data
N_sample.to_csv("~/MA/data/bt_augmented_agg_pub/.csv", sep='\t', index=False, quoting=1)